# Eminescu — anatomia unei prezențe ubicue

Mihai Eminescu nu este doar cel mai onorat poet român pe străzi. Este unul dintre cei mai onorați oameni, indiferent de profesie. Acest notebook măsoară amploarea exactă și o compară cu alți scriitori canonici.

Sursa: Registrul Secțiilor de Vot · Clasificare: curation manual + LLM batch

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
ACCENT = '#C04F35'
INK    = '#15171A'
MUTED  = '#6E6E70'

conn = sqlite3.connect('../data/streets.db')
conn.row_factory = sqlite3.Row

# În tabela `persons` Eminescu apare cu două chei (mihai eminescu / mihail eminescu).
# Pentru analiză le tratăm ca pe o singură persoană.
EMINESCU_KEYS = ('mihai eminescu', 'mihail eminescu')

## 1. Cifra de bază

Câte străzi îl onorează pe Eminescu, în câte UAT-uri, și ce înseamnă asta raportat la totalul de UAT-uri din țară?

In [ ]:
headline = pd.read_sql(f"""
    SELECT COUNT(*) AS streets,
           COUNT(DISTINCT siruta) AS uats_with_eminescu
    FROM streets_dedup
    WHERE core_name_norm IN {EMINESCU_KEYS}
""", conn)

total_uats = pd.read_sql("SELECT COUNT(DISTINCT siruta) AS total FROM streets_dedup", conn)['total'][0]

streets   = headline['streets'][0]
uats      = headline['uats_with_eminescu'][0]
uat_share = uats / total_uats * 100

print(f'Străzi:              {streets:>5}')
print(f'UAT-uri acoperite:   {uats:>5}')
print(f'Total UAT-uri:       {total_uats:>5}')
print(f'')
print(f'Eminescu apare în {uat_share:.1f}% din UAT-urile României.')

## 2. Comparație cu egalii

Cum se așază Eminescu între ceilalți scriitori și poeți canonici?

In [ ]:
peers = pd.read_sql("""
    SELECT p.full_name, p.profession, p.era,
           COUNT(*) AS streets,
           COUNT(DISTINCT sd.siruta) AS uats
    FROM streets_dedup sd
    JOIN persons p ON p.core_name_norm = sd.core_name_norm
    WHERE p.profession IN ('poet','writer')
      AND p.era IN ('premodern','1848','interwar')
    GROUP BY p.full_name
    ORDER BY streets DESC
    LIMIT 12
""", conn)

# Două intrări Eminescu apar separat — le consolidăm pentru claritate
peers = peers.groupby(['full_name','profession','era'], as_index=False).agg(
    streets=('streets','sum'), uats=('uats','sum')
).sort_values('streets', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 6))
colors = [ACCENT if 'Eminescu' in n else INK for n in peers['full_name']]
ax.barh(peers['full_name'][::-1], peers['streets'][::-1],
        color=colors[::-1], alpha=0.9)
for i, (n, s) in enumerate(zip(peers['streets'][::-1], peers['era'][::-1])):
    ax.text(n + 5, i, f'{n}  ·  {s}', va='center', fontsize=9, color=MUTED)
ax.set_title('Cei mai onorați scriitori și poeți români', fontsize=13, pad=10)
ax.set_xlabel('Număr de străzi')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'.replace(',','.')))
plt.tight_layout()
plt.show()

Plumb-line-ul lui Eminescu nu e doar primul loc — e o distanță sensibilă față de locul doi. Creangă (best friend, biograf, complice de viață) e cel mai apropiat. Coșbuc, al treilea, e încă din aceeași generație.

## 3. Distribuția geografică

În care județe e Eminescu mai prezent? Dat fiind că este universal, întrebarea interesantă e *unde este mai dens decât media națională*.

In [ ]:
by_judet = pd.read_sql(f"""
    SELECT sd.judet,
           COUNT(*) AS eminescu_streets,
           (SELECT COUNT(DISTINCT siruta)
              FROM streets_dedup WHERE judet = sd.judet) AS uats_in_judet
    FROM streets_dedup sd
    WHERE sd.core_name_norm IN {EMINESCU_KEYS}
    GROUP BY sd.judet
    ORDER BY eminescu_streets DESC
""", conn)
by_judet['density'] = by_judet['eminescu_streets'] / by_judet['uats_in_judet']

top15 = by_judet.head(15)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(top15['judet'], top15['eminescu_streets'], color=INK, alpha=0.85, label='Străzi')
ax2 = ax.twinx()
ax2.plot(top15['judet'], top15['density'], color=ACCENT, marker='o', linewidth=1.8, label='Densitate (străzi / UAT)')
ax2.spines['top'].set_visible(False)
ax.set_title('Eminescu pe județ — număr absolut vs. densitate', fontsize=13)
ax.set_ylabel('Număr de străzi', color=INK)
ax2.set_ylabel('Densitate (str. / UAT)', color=ACCENT)
ax.tick_params(axis='x', rotation=0)
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.show()

## 4. UAT-uri unde Eminescu lipsește

În aproximativ două treimi din UAT-uri nu există nicio stradă numită Eminescu. Sunt acestea concentrate într-o regiune anume sau distribuite uniform?

In [ ]:
missing = pd.read_sql(f"""
    WITH eminescu_uats AS (
      SELECT DISTINCT siruta FROM streets_dedup
      WHERE core_name_norm IN {EMINESCU_KEYS}
    ),
    all_uats AS (
      SELECT DISTINCT judet, siruta, uat FROM streets_dedup
    )
    SELECT a.judet,
           COUNT(*) AS total_uats,
           SUM(CASE WHEN e.siruta IS NULL THEN 1 ELSE 0 END) AS missing_uats,
           ROUND(SUM(CASE WHEN e.siruta IS NULL THEN 1.0 ELSE 0 END)
                 / COUNT(*) * 100, 1) AS pct_missing
    FROM all_uats a
    LEFT JOIN eminescu_uats e USING(siruta)
    GROUP BY a.judet
    HAVING total_uats >= 10
    ORDER BY pct_missing DESC
""", conn)

fig, ax = plt.subplots(figsize=(12, 5))
colors = [ACCENT if p > 50 else (INK if p > 30 else MUTED) for p in missing['pct_missing']]
ax.bar(missing['judet'], missing['pct_missing'], color=colors)
ax.axhline(missing['pct_missing'].median(), color=INK, linestyle='--', linewidth=0.8,
           label=f'Mediană: {missing["pct_missing"].median():.1f}%')
ax.set_title('% UAT-uri fără nicio stradă Eminescu, pe județ', fontsize=13)
ax.set_ylabel('% UAT-uri fără Eminescu')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Județele cu cele mai puține lipsuri sunt indicatori ai unei *acoperiri* near-totale. Cele cu lipsuri majore tind să fie zone rurale unde nomenclatorul stradal este dominat de denumiri din natură (Principală, Bisericii, Viilor) — nu prezența lui Eminescu e neașteptată, ci absența numelor de persoane în general.

## 5. Variațiile numelui

Cum este scris Eminescu pe străzi? Există o normă oficială ("Mihai Eminescu"), dar registrul electoral surprinde varianta locală a fiecărei UAT.

In [ ]:
variants = pd.read_sql(f"""
    SELECT name_normalized AS form, COUNT(*) AS streets
    FROM streets_dedup
    WHERE core_name_norm IN {EMINESCU_KEYS}
    GROUP BY name_normalized
    ORDER BY streets DESC
    LIMIT 15
""", conn)
variants

## 6. Sinteză

Eminescu este cel mai onorat poet, cel mai onorat scriitor și — alături de Cuza, Eroii, Mihai Viteazul — una dintre cele mai onorate cinci persoane în nomenclatura străzilor din România. Distribuția lui este near-uniformă: nu există județ în care să lipsească complet, dar există județe rurale în care prezența numelor de persoane este oricum scăzută.

Două extensii naturale ale acestei analize:

1. **Lungimea străzilor Eminescu vs. media** — folosind tabela `osm_streets`, am putea verifica dacă Eminescu primește bulevarde sau străduțe.
2. **Eminescu vs. Sadoveanu vs. Caragiale într-un singur grafic regional** — care e "poetul fiecărei regiuni"?

In [ ]:
conn.close()